In [11]:
import tensorflow as tf
import numpy as np
import pandas as pd

### **Carga y limpieza de datos:**

En esta segunda prueba para la generación de una red neuronal para la predicción de ataques al corazon, los datos provienen con un mejor balance, lo cual permite al modelo interpretar mejor las caracteristicas y llegar a una inferencia.

la mayor parte de los datos son de tipo numerico, no existen datos irrelevantes y los pocos de tipo caracter son mapeados a numeros de forma categorica y la salida del modelo debe ser la columna  **HeartDisease**, en la cual podremos predecir el desceso del paciente.

In [12]:
df=pd.read_csv('Datasets/heart.csv')

for i in df.columns:
    if df[i].dtype== 'object' or df[i].dtype== 'str':
        df[i]=pd.Categorical(df[i])
        df[i]=df[i].cat.codes

print(df.dtypes)

df.head(2)

Age                 int64
Sex                  int8
ChestPainType        int8
RestingBP           int64
Cholesterol         int64
FastingBS           int64
RestingECG           int8
MaxHR               int64
ExerciseAngina       int8
Oldpeak           float64
ST_Slope             int8
HeartDisease        int64
dtype: object


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,1,1,140,289,0,1,172,0,0.0,2,0
1,49,0,2,160,180,0,1,156,0,1.0,1,1


El 55% de los datos indican muerte por ataque al corazon, y el 45% no mueren por esta causa, los datos se encuentran mejor nivelados y se puede realizar un mejor proceso.

In [13]:
col_output='HeartDisease'
a=len(df[df[col_output]==1])
b=len(df[df[col_output]==0])
print(a,b, 100*a/(a+b))

508 410 55.33769063180828


### **Datos de ensayo y verificación**

Se genera una columna de numeros aleatorios, la cual se discretiza, en caso de que sea menor que 4, entran en el modelo mayor, en caso contrario, en el modelo de prueba, el cual se va a usar mas adelante para evaluar el rendimiento del modelo (Se puede dividir el modelo en mas partes, pero con esto es suficiente).

In [14]:
rand_sel=np.random.rand(len(df))*5
rand_sel=rand_sel.astype(int)
df['random']=rand_sel
df_princ=df[df['random']<4].copy().drop(columns='random')
df_1=df[df['random']==4].copy().drop(columns='random')
df_2=df[df['random']==5].copy().drop(columns='random')
df_3=df[df['random']==6].copy().drop(columns='random')

Las proporciones de la población no varian mucho entre sus valores de salida, aproximadamenteel 50% de los datos son negativos y el 50% son positivos.

In [15]:
print(  len(df_princ[df_princ[col_output]==1]),
        len(df_princ[df_princ[col_output]==0]), '\n',
        len(df_1[df_1[col_output]==1]), 
        len(df_1[df_1[col_output]==0]), '\n',
        len(df_2[df_2[col_output]==1]), 
        len(df_2[df_2[col_output]==0]), '\n',
        len(df_3[df_3[col_output]==1]),
        len(df_3[df_3[col_output]==0]))

390 320 
 118 90 
 0 0 
 0 0


### **Definición del modelo:**

Se procede a definir el modelo, con los datos categorizados y la unica salida binaria

In [16]:
x=[]
for i in df_princ.columns:
    if i!= col_output:
        x.append(df_princ[i])

X=np.column_stack(x)
tamaño= len(x)

Se puede usar un numero de capas adicionales para mejorar el rendimiento del modelo, pero con una sola se obtienen buenas clasificaciones.

In [17]:
entrada = tf.keras.layers.Dense(units=tamaño, input_shape=[tamaño])
c1 = tf.keras.layers.Dense(units=tamaño)
c2 = tf.keras.layers.Dense(units=tamaño)
c3 = tf.keras.layers.Dense(units=tamaño)
salida = tf.keras.layers.Dense(units=1, activation='sigmoid')
red = tf.keras.Sequential([entrada, c1, c2, salida])
#red = tf.keras.Sequential([entrada, c1, c2,c3,c4, c5,c6, salida])
red.compile(optimizer='adam',
            loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
            metrics=['accuracy'])

In [18]:
historial = red.fit(X, df_princ[col_output], epochs=1000, verbose=False)

/home/robotica/Documentos/IA-minirobots/S5 redes neuronales/Trabajo/IA_minibots_T5_redes_neuronales/.venv/lib/python3.8/site-packages/keras/src/backend.py:5805: UserWarning: "`binary_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Sigmoid activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


### Prueba de red neuronal para predecir ataques cardíacos

Se genera una función de pandas para clasificar cada fila de datos en el modelo de red neuronal, y se discrimina con 0.5 para llegar a una clasificación binaria.

In [19]:

def clasificar_dataframe(df, modelo, columnas):
    X = df[columnas].values
    preds = modelo.predict(X)
    df['clasificacion'] = preds
    df['Predeccion']=(preds > 0.5).astype(int)
    return df

cols_ver=df_1.columns.tolist()
cols_ver.remove(col_output)

df_1= clasificar_dataframe(df_1, red, cols_ver)
df_1.head(2)

7/7 [==============================] - 0s 660us/step


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease,clasificacion,Predeccion
6,45,0,1,130,237,0,1,170,0,0.0,2,0,0.037366,0
14,42,0,2,115,211,0,2,137,0,0.0,2,0,0.016516,0


### Matriz de confusión

Al evaluar el comportamiento de la red neuronal, con un modelo externo (df_1), se puede observar que el modelo predice correctamente el 86% de los casos, con mas valores de falsos positivos que de falsos negativos.

In [20]:
cm = pd.crosstab(df_1[col_output], df_1['Predeccion'])

TN = cm.loc[0, 0]
FP = cm.loc[0, 1]
FN = cm.loc[1, 0]
TP = cm.loc[1, 1]
accuracy = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP)
recall = TP / (TP + FN)
f1 = 2 * (precision * recall) / (precision + recall)
print(accuracy)
cm

0.8605769230769231


Predeccion,0,1
HeartDisease,,
0,72,18
1,11,107
